# Notebook 04 — JWT auth

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

Why switch the registry from IAM to JWT, and what does it enable?

This notebook answers that question with working code. By the end, you will have
seen how JWT-based auth carries the persona claim as a token claim — and you will
understand why this enables per-request authorization without IAM role assumption
per user.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
# Skips uv sync (which installs the full agent stack and takes minutes).
import sys, subprocess

# Only install packages not already provided by the SageMaker base image
pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0']

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **IAM-based auth** | AWS Identity and Access Management authentication where a service assumes a role to call another service. Phase 1 uses this: the UI's backend assumes an IAM role to call the registry. The role determines what the service can do, not who the user is. |
| **JWT-based auth** | JSON Web Token authentication where a user's identity and claims are encoded in a signed token. Phase 2 uses this: the user's browser sends a JWT to the registry, and the registry reads the persona claim directly from the token. |
| **Persona claim** | A custom claim in the JWT payload (e.g., `custom:persona = atlas-wealth-advisor`) that identifies the user's role. The registry uses this claim to filter which capabilities are returned. |
| **Cognito** | AWS Cognito — the identity provider that issues JWTs for ATLAS users. Each user pool has custom attributes that map to persona claims. |
| **Token claim** | A key-value pair in the JWT payload. Standard claims include `sub` (subject), `iss` (issuer), `exp` (expiration). Custom claims like `custom:persona` carry application-specific data. |
| **Fine-grained authorization** | Authorization that operates at the individual request level based on token claims, rather than at the service level based on IAM roles. Each API call carries its own authorization context. |

## From service identity to user identity

Phase 1 uses IAM-based authentication. The Wholesale UI's backend is a Lambda
function that assumes an IAM role — `atlas-wholesale-ui-role` — to call the
registry and MCP servers. The IAM role grants permission to invoke those services,
but it does not carry information about which user is making the request. The
persona claim is passed as a parameter in the request body: the UI code sets
`persona_claim: "atlas-consumer-banker"` because it knows all its users are
Consumer Bankers. This works for Phase 1 because there is only one UI and one
persona. The IAM role is the authorization boundary.

Phase 2 breaks this model. Two UIs serve two different personas, and both call
the same registry endpoint. If both UIs use IAM roles, the registry cannot
distinguish between a Consumer Banker request and a Wealth Advisor request at
the IAM level — both come from valid service roles. The persona claim would
still be a request parameter, which means the UI code decides what persona to
claim. A misconfigured or compromised UI could claim any persona. IAM does not
prevent this because IAM authorizes the service, not the user.

JWT-based auth solves this by moving the persona claim from the request body into
the token itself. When a user logs in through Cognito, the identity provider
issues a JWT that includes `custom:persona` as a signed claim. The registry reads
this claim directly from the token — it does not trust the request body. Because
the token is signed by Cognito, the claim cannot be forged by the UI code. The
authorization decision happens at the token level, not the service level: each
request carries its own identity and its own persona, verified by cryptographic
signature.

This also eliminates the need for IAM role assumption per user. In an IAM-only
model, supporting N personas would require N IAM roles and N role-assumption
paths. In the JWT model, all users authenticate through the same Cognito user
pool, receive tokens with different persona claims, and call the same endpoint.
The registry filters by the claim in the token. Adding a new persona requires
adding a new claim value in Cognito and updating the registry's filter logic —
no new IAM roles, no new trust policies, no new role-assumption code.

In [ ]:
import sys
import os
import json
import base64
import time

# Workshop 1's shared helpers.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

from pathlib import Path

SPEC_DIR = "../../spec/04-aws-agent-registry"

def load_descriptors(subdir):
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    if not os.path.isdir(path):
        print(f"WARNING: {path} not found.")
        return descriptors
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

agent_descriptors = load_descriptors("agents")

print(f"Agents loaded: {len(agent_descriptors)}")
print("Setup complete.")

In [ ]:
# Build cell 1 — Demonstrate JWT token structure with persona claim.
#
# A real JWT has three parts: header.payload.signature
# We construct a sample payload showing the persona claim.

def create_sample_jwt_payload(persona, sub="user-12345"):
    """Create a sample JWT payload with persona claim."""
    now = int(time.time())
    return {
        "sub": sub,
        "iss": "https://cognito-idp.us-east-1.amazonaws.com/us-east-1_AtlasPool",
        "aud": "atlas-app-client-id",
        "exp": now + 3600,
        "iat": now,
        "custom:persona": persona,
        "custom:team_id": "wealth-team-east",
        "token_use": "id",
    }

# Consumer Banker token
banker_payload = create_sample_jwt_payload("atlas-consumer-banker")
print("Consumer Banker JWT payload:")
print(json.dumps(banker_payload, indent=2))
print()

# Wealth Advisor token
advisor_payload = create_sample_jwt_payload("atlas-wealth-advisor")
print("Wealth Advisor JWT payload:")
print(json.dumps(advisor_payload, indent=2))
print()

print("Key difference: custom:persona claim identifies the user's role.")
print("The registry reads this claim to filter capabilities.")

In [ ]:
# Build cell 2 — Show how registry filters by JWT claim.
#
# The registry extracts custom:persona from the JWT and uses it
# to filter which agents are returned. No request body parameter needed.

def registry_resolve_from_jwt(jwt_payload, descriptors):
    """Simulate registry filtering by JWT persona claim.
    
    In production, AppSync extracts the claim from the Authorization
    header and passes it to the resolver. The resolver filters agents
    by discoverable_by matching the claim value.
    """
    persona = jwt_payload.get("custom:persona")
    if not persona:
        return {"error": "No persona claim in token", "capabilities": []}
    
    capabilities = [
        d["agent_name"]
        for d in descriptors
        if persona in d.get("registry_metadata", {}).get("discoverable_by", [])
    ]
    return {"persona": persona, "capabilities": capabilities}

# Filter using banker token
banker_result = registry_resolve_from_jwt(banker_payload, agent_descriptors)
print(f"Registry result for Consumer Banker JWT:")
print(f"  Persona: {banker_result['persona']}")
print(f"  Capabilities: {banker_result['capabilities']}")
print()

# Filter using advisor token
advisor_result = registry_resolve_from_jwt(advisor_payload, agent_descriptors)
print(f"Registry result for Wealth Advisor JWT:")
print(f"  Persona: {advisor_result['persona']}")
print(f"  Capabilities: {advisor_result['capabilities']}")
print()

# Token without persona claim
bad_payload = {"sub": "user-99999", "exp": int(time.time()) + 3600}
bad_result = registry_resolve_from_jwt(bad_payload, agent_descriptors)
print(f"Registry result for token without persona claim:")
print(f"  {bad_result}")

In [ ]:
# Build cell 3 — Compare IAM vs JWT authorization models.

comparison = {
    "Phase 1 (IAM)": {
        "auth_method": "IAM role assumption",
        "persona_source": "Request body parameter",
        "trust_model": "Trust the calling service",
        "per_user_cost": "One IAM role per persona",
        "forgery_risk": "UI code can claim any persona",
    },
    "Phase 2 (JWT)": {
        "auth_method": "Cognito JWT token",
        "persona_source": "Signed token claim",
        "trust_model": "Trust the identity provider",
        "per_user_cost": "One token claim value",
        "forgery_risk": "Claim is cryptographically signed",
    },
}

print("IAM vs JWT authorization comparison:")
print("=" * 60)
for phase, details in comparison.items():
    print(f"\n{phase}:")
    for key, value in details.items():
        print(f"  {key:<20} {value}")

## Verification

Two properties must hold: the JWT contains the persona claim (so the registry can
read it without trusting the request body), and the registry respects JWT-based
filtering (returning different results for different persona claims in the token).
If either fails, the two-UI model cannot enforce persona boundaries at the request
level.

In [ ]:
# Verification cell 1 — JWT contains persona claim.

print("Verifying JWT persona claim structure...")
print()

# Test both persona tokens
test_personas = ["atlas-consumer-banker", "atlas-wealth-advisor"]
claim_failures = []

for persona in test_personas:
    payload = create_sample_jwt_payload(persona)
    has_claim = "custom:persona" in payload
    claim_value = payload.get("custom:persona")
    correct_value = claim_value == persona
    
    status = "[PASS]" if (has_claim and correct_value) else "[FAIL]"
    print(f"  {status} {persona}: custom:persona = {claim_value}")
    
    if not (has_claim and correct_value):
        claim_failures.append(persona)

print()

if claim_failures:
    print(f"VERIFICATION FAILED: Missing or incorrect persona claim for: {claim_failures}")
    print("The JWT payload must include 'custom:persona' with the correct persona value.")
    print("Check the Cognito user pool custom attributes configuration.")

assert not claim_failures, (
    f"JWT persona claim missing or incorrect for: {claim_failures}. "
    "Each token must carry custom:persona with the user's assigned persona."
)

print("[PASS] All JWT tokens contain the correct persona claim.")

In [ ]:
# Verification cell 2 — Registry respects JWT-based filtering.

print("Verifying registry JWT-based filtering...")
print()

# Different tokens must produce different capability sets
banker_caps = set(registry_resolve_from_jwt(
    create_sample_jwt_payload("atlas-consumer-banker"), agent_descriptors
)["capabilities"])

advisor_caps = set(registry_resolve_from_jwt(
    create_sample_jwt_payload("atlas-wealth-advisor"), agent_descriptors
)["capabilities"])

print(f"Consumer Banker capabilities: {sorted(banker_caps)}")
print(f"Wealth Advisor capabilities:  {sorted(advisor_caps)}")
print()

# Token without persona must return empty
no_persona_result = registry_resolve_from_jwt(
    {"sub": "user-anon", "exp": int(time.time()) + 3600}, agent_descriptors
)
no_persona_caps = no_persona_result.get("capabilities", [])
print(f"No-persona token capabilities: {no_persona_caps}")
print()

filtering_works = (
    banker_caps != advisor_caps
    and len(banker_caps) > 0
    and len(advisor_caps) > 0
    and len(no_persona_caps) == 0
)

if not filtering_works:
    print("VERIFICATION FAILED: Registry does not properly filter by JWT claim.")
    print("Expected: different non-empty sets for different personas,")
    print("and empty set for missing persona claim.")

assert filtering_works, (
    "Registry must return different capabilities for different JWT persona claims, "
    "and empty capabilities for tokens without a persona claim."
)

print("[PASS] Registry correctly filters by JWT persona claim.")
print("Per-request authorization works without IAM role assumption per user.")

## What just changed

You have seen why Phase 2 switches from IAM-based auth to JWT-based auth. The
persona claim moves from a request body parameter (trusted by convention) to a
signed token claim (trusted by cryptographic verification). This enables per-request
authorization: each API call carries its own identity, and the registry filters
capabilities based on the claim in the token rather than the IAM role of the
calling service.

The next notebook walks the full advisor scenario across both UIs — from signal
detection in the Wholesale UI to conversational follow-up in the Wealth UI —
showing how JWT auth, memory, and behavioral signals compose into a complete
cross-persona workflow.